#Random Label Machine Unlearning with Retain Loss (v2)

Same random-label forget mechanism as v1, but adds a **retain loss** on non-target passages
to prevent catastrophic forgetting of neighbour knowledge.

In [1]:
!pip install -q transformers peft trl datasets accelerate


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from utils2 import load_rwku_data, prepare_tokenized_dataset, evaluate_model, evaluate_neighbours

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/Users/Hania/Documents/mul_for_llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [3]:
# downloanding the model
MODEL_ID = "Qwen/Qwen2.5-3B"

SUBJECT   = "Donald Trump"

print(f"Loading model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
)
model = model.to(DEVICE)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
print("Number of parameters for training:")
peft_model.print_trainable_parameters()

Loading model: Qwen/Qwen2.5-3B


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:05<00:00, 75.06it/s] 


Number of parameters for training:
trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


In [4]:
person_train, questions_forget, keywords_forget, questions_retain, keywords_retain = load_rwku_data("Donald Trump")
tokenized_forget_dataset = prepare_tokenized_dataset(person_train, tokenizer)

Questions for forgetting test: 20
Questions for general knowledge test: 30
Texts for training (unlearning): 226
Data is ready


In [5]:
print("BASELINE: model knowledge BEFORE unlearning")

print("EFFICACY — direct questions about Donald Trump (should be HIGH)")
acc_forget_before = evaluate_model(peft_model, tokenizer, questions_forget, keywords_forget, DEVICE)

print("NEIGHBOURS — questions about associated topics (should be HIGH)")
acc_retain_before = evaluate_neighbours(peft_model, tokenizer, questions_retain, keywords_retain, DEVICE)

BASELINE: model knowledge BEFORE unlearning
EFFICACY — direct questions about Donald Trump (should be HIGH)
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the answer is "the apprentice."'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and hosted the reality television series ___.
Expected: 'the apprentice'
Model generated: 'the answer is "the apprentice."'
Result: PASSED

--------------

In [7]:
# Build retain dataset from RWKU training passages for subjects other than the target.
# These passages are used in the retain loss pass so the model keeps general knowledge
# intact while the random-label pass pushes it away from target-specific knowledge.
from datasets import load_dataset as _load_ds

_all_train = _load_ds("jinzhuoran/RWKU", 'train_original_passage', split='train')
retain_raw = _all_train.filter(lambda x: SUBJECT not in x['subject'])
retain_raw = retain_raw.select(range(min(300, len(retain_raw))))
tokenized_retain_dataset = prepare_tokenized_dataset(retain_raw, tokenizer)
print(f"Retain dataset: {len(tokenized_retain_dataset)} passages (subjects other than {SUBJECT})")

Data is ready
Retain dataset: 300 passages (subjects other than Donald Trump)


In [8]:
import os
os.environ["TQDM_DISABLE"] = "1"

from torch.utils.data import DataLoader
from transformers import TrainerCallback

class PrintProgress(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero and logs and "loss" in logs:
            print(f"  step {state.global_step}/{state.max_steps}  loss={logs['loss']:.4f}")

def _collate_retain(batch):
    """Convert HuggingFace dataset rows to stacked tensors for the retain DataLoader."""
    return {
        'input_ids':      torch.stack([torch.tensor(b['input_ids'])      for b in batch]),
        'attention_mask': torch.stack([torch.tensor(b['attention_mask']) for b in batch]),
        'labels':         torch.stack([torch.tensor(b['labels'])         for b in batch]),
    }

class RandomLabelTrainer(Trainer):
    def __init__(self, *args, retain_dataset=None, beta=1.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta
        if retain_dataset is not None:
            self.retain_loader = DataLoader(
                retain_dataset,
                batch_size=1,
                shuffle=True,
                collate_fn=_collate_retain,
            )
            self._retain_iter = iter(self.retain_loader)
        else:
            self.retain_loader = None

    def _next_retain_batch(self):
        try:
            return next(self._retain_iter)
        except StopIteration:
            self._retain_iter = iter(self.retain_loader)
            return next(self._retain_iter)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs = {k: v.clone() for k, v in inputs.items()}
        inputs['labels'] = torch.randint(
            0, model.config.vocab_size,
            inputs['labels'].shape,
            device=inputs['labels'].device
        )
        forget_out  = model(**inputs)
        forget_loss = forget_out.loss

        total_loss = forget_loss
        if self.retain_loader is not None:
            retain_batch = {k: v.to(model.device)
                            for k, v in self._next_retain_batch().items()}
            retain_out  = model(**retain_batch)
            total_loss  = forget_loss + self.beta * retain_out.loss

        return (total_loss, forget_out) if return_outputs else total_loss

training_args = TrainingArguments(
    output_dir="./unlearning_results_rl_v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    max_steps=100,
    logging_steps=1,
    gradient_checkpointing=False,
    optim="adamw_torch",
    report_to="none",
)

trainer = RandomLabelTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_forget_dataset,
    retain_dataset=tokenized_retain_dataset,
    beta=1.5,
    callbacks=[PrintProgress()],
)

print("Unlearning started (Random Labels + Retain Loss)")
trainer.train()
print("Finished")

save_path = "./unlearned_model_rl_qwen3-4B_v2"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model and tokenizer saved to {save_path}")

Unlearning started (Random Labels + Retain Loss)


/Users/Hania/Documents/mul_for_llm/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,59.506260
2,51.140701
3,57.273331
4,53.311951
5,60.585976
6,54.094265
7,54.630302
8,62.686783
9,58.971340
10,60.087749


  step 1/100  loss=59.5063
  step 2/100  loss=51.1407
  step 3/100  loss=57.2733
  step 4/100  loss=53.3120
  step 5/100  loss=60.5860
  step 6/100  loss=54.0943
  step 7/100  loss=54.6303
  step 8/100  loss=62.6868
  step 9/100  loss=58.9713
  step 10/100  loss=60.0877
  step 11/100  loss=52.0689
  step 12/100  loss=52.4672
  step 13/100  loss=53.4320
  step 14/100  loss=59.7210
  step 15/100  loss=51.1428
  step 16/100  loss=57.1162
  step 17/100  loss=58.3430
  step 18/100  loss=50.1384
  step 19/100  loss=52.8775
  step 20/100  loss=60.7381
  step 21/100  loss=50.9334
  step 22/100  loss=54.1210
  step 23/100  loss=57.3580
  step 24/100  loss=55.2746
  step 25/100  loss=56.5223
  step 26/100  loss=51.5185
  step 27/100  loss=53.5449
  step 28/100  loss=52.9871
  step 29/100  loss=47.5893
  step 30/100  loss=55.0939
  step 31/100  loss=53.6617
  step 32/100  loss=52.5717
  step 33/100  loss=47.1346
  step 34/100  loss=55.0529
  step 35/100  loss=50.2813
  step 36/100  loss=53.8844
 

In [9]:
print("AFTER unlearning (v2: Random Labels + Retain Loss)")

print("EFFICACY — direct questions about Donald Trump")
acc_forget_after = evaluate_model(peft_model, tokenizer, questions_forget, keywords_forget, DEVICE)

print("NEIGHBOURS — questions about associated topics")
acc_retain_after = evaluate_neighbours(peft_model, tokenizer, questions_retain, keywords_retain, DEVICE)

print("\nSUMMARY")
print(f"Efficacy   (Donald Trump direct) — before: {acc_forget_before:.1f}%  |  after: {acc_forget_after:.1f}%")
print(f"Neighbours (general)        — before: {acc_retain_before:.1f}%  |  after: {acc_retain_after:.1f}%")

AFTER unlearning (v2: Random Labels + Retain Loss)
EFFICACY — direct questions about Donald Trump
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the answer is "the apprentice".'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and hosted the reality television series ___.
Expected: 'the apprentice'
Model generated: 'the answer is "the apprentice".'
Result: PASSED

------------------------